# 03 — Option blotter

Run `python scripts/run_backtest.py` first. P&L is a SABR/VIX mark, not an NSE fill.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))

import pandas as pd
from config import DUCKDB_PATH, PROCESSED_DATA_PATH
from data_pipeline.db_utils import DatabaseManager

def load(name):
    pq = PROCESSED_DATA_PATH / f"{name}.parquet"
    csv = PROCESSED_DATA_PATH / f"{name}.csv"
    if pq.exists():
        return pd.read_parquet(pq)
    if csv.exists():
        return pd.read_csv(csv)
    db = DatabaseManager(DUCKDB_PATH)
    if db.table_exists(name):
        return db.load_table(name).to_pandas()
    return pd.DataFrame()

t = load("option_trades")
print("trades", len(t))
if not t.empty:
    t["date_out"] = pd.to_datetime(t["date_out"])
    t["year"] = t["date_out"].dt.year
    print(t.groupby("year").agg(n=("win", "size"), win=("win", "mean"), pnl=("pnl_inr", "sum")))
    if "action" in t.columns:
        print(t.groupby("action").agg(n=("win", "size"), win=("win", "mean"), pnl=("pnl_inr", "sum")))


In [ ]:
if not t.empty:
    import matplotlib.pyplot as plt
    eq = t.sort_values("date_out")
    fig, ax = plt.subplots(figsize=(12, 4))
    if "equity_after" in eq.columns:
        ax.plot(eq["date_out"], eq["equity_after"], color="#1a365d", lw=1.4)
        ax.set_ylabel("Equity (INR)")
    else:
        ax.plot(eq["date_out"], eq["pnl_inr"].cumsum(), color="#1a365d", lw=1.4)
        ax.set_ylabel("Cumulative P&L (INR)")
    ax.set_title("Options blotter equity path")
    plt.tight_layout()
    plt.show()


In [ ]:
from execution.strategy import decide, prepare_features
reg = load("regime_predictions")
if not reg.empty:
    reg = prepare_features(reg)
    last = decide(reg.iloc[-1])
    print(last)
